# DTM: Temporal Topic Analysis (Native)

DTM is **natively temporal** — uses `get_topic_word_dist(topic, timepoint)` directly.

In [1]:
import ast
import time
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import tomotopy as tp
import warnings
warnings.filterwarnings("ignore")

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
BASE_DIR = Path("../../../../data/preprocess")
MODEL_DIR = Path("../../../../models/dtm/tuning")
RESULT_DIR = Path("../../../../results/dtm/temporal")
VERSION = "v1"
TOP_N_WORDS = 10
RBO_P = 0.9

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

In [3]:
def get_topic_words_at_timepoint(model, timepoint, top_n=10):
    vocab = list(model.used_vocabs)
    result = {}
    for tid in range(model.k):
        dist = model.get_topic_word_dist(tid, timepoint=timepoint)
        top_idx = np.argsort(dist)[::-1][:top_n]
        result[tid] = [vocab[i] for i in top_idx]
    return result


def get_aggregated_topic_words(model, top_n=10):
    vocab = list(model.used_vocabs)
    result = {}
    for tid in range(model.k):
        avg_dist = np.zeros(len(vocab))
        for t in range(model.num_timepoints):
            avg_dist += np.array(model.get_topic_word_dist(tid, timepoint=t))
        avg_dist /= model.num_timepoints
        top_idx = np.argsort(avg_dist)[::-1][:top_n]
        result[tid] = [vocab[i] for i in top_idx]
    return result


def rbo(list1, list2, p=0.9):
    if not list1 and not list2:
        return 1.0
    if not list1 or not list2:
        return 0.0

    # assign short (S) and long (L)
    if len(list1) <= len(list2):
        S, L = list1, list2
    else:
        S, L = list2, list1

    s, l = len(S), len(L)

    S_seen = set()
    L_seen = set()

    X = 0  # overlap
    rbo = 0.0
    disjoint = 0.0
    ext_term = 0.0

    for d in range(l):
        if d < s:
            s_item = S[d]
            S_seen.add(s_item)
        else:
            s_item = None

        l_item = L[d]
        L_seen.add(l_item)

        overlap_incr = 0

        if d < s:
            if s_item == l_item:
                overlap_incr = 1
            else:
                if s_item in L_seen:
                    overlap_incr += 1
                if l_item in S_seen:
                    overlap_incr += 1
        else:
            if l_item in S_seen:
                overlap_incr = 1

        X += overlap_incr

        if d < s:
            A_d = 2.0 * X / (len(S_seen) + len(L_seen))
        else:
            A_d = X / (d + 1)

        rbo += (1 - p) * (p ** d) * A_d

        if d < s:
            ext_term = A_d * (p ** (d + 1))
        else:
            X_s = X - overlap_incr if d == s else X_s
            disjoint += (1 - p) * (p ** d) * (
                X_s * (d + 1 - s) / ((d + 1) * s)
            )
            ext_term = (
                ((X - X_s) / (d + 1) + X_s / s)
                * (p ** (d + 1))
            )

        # optional optimization (safe)
        if p ** d < 1e-12:
            break

    return min(max(rbo + disjoint + ext_term, 0.0), 1.0)

def calculate_irbo(topics_words_list, p=0.9):
    if len(topics_words_list) < 2:
        return 0.0
    scores = [1.0 - rbo(topics_words_list[i], topics_words_list[j], p)
              for i, j in combinations(range(len(topics_words_list)), 2)]
    return np.mean(scores)

## Load Models & Data

In [4]:
all_models = {}
all_data = {}
all_years = {}
all_year_to_tp = {}

for subject in LIST_SUBJECT:
    print(f"\nLoading {subject}...")

    model = tp.DTModel.load(str(MODEL_DIR / subject / "best_model.bin"))
    all_models[subject] = model

    df = pd.read_csv(BASE_DIR / subject / "bow" / f"{VERSION}.csv")
    df["submitted_date"] = pd.to_datetime(df["submitted_date"])
    df["year"] = df["submitted_date"].dt.year

    # Filter empty docs (DTM skips them during training)
    df["tokens"] = df["text"].apply(lambda x: ast.literal_eval(x))
    df = df[df["tokens"].apply(len) > 0].reset_index(drop=True)

    years = sorted(df["year"].unique())
    year_to_tp = {year: i for i, year in enumerate(years)}
    all_years[subject] = years
    all_year_to_tp[subject] = year_to_tp

    print(f"  Model docs: {len(model.docs)}, DataFrame: {len(df)}")
    topics = [int(np.argmax(doc.get_topic_dist())) for doc in model.docs]
    df["topic"] = topics

    all_data[subject] = df
    print(f"  {subject}: {len(df):,} docs, {model.k} topics, "
          f"{len(years)} years ({years[0]}-{years[-1]})")

print(f"\n✅ All subjects loaded")


Loading cs...
  Model docs: 165756, DataFrame: 165756
  cs: 165,756 docs, 50 topics, 26 years (2000-2025)

Loading math...
  Model docs: 157084, DataFrame: 157084
  math: 157,084 docs, 50 topics, 26 years (2000-2025)

Loading physics...
  Model docs: 146311, DataFrame: 146311
  physics: 146,311 docs, 50 topics, 26 years (2000-2025)

✅ All subjects loaded


## Topic Prevalence Over Time

In [5]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    global_tw = get_aggregated_topic_words(model, top_n=5)

    rows = []
    for year in years:
        year_df = df[df["year"] == year]
        counts = year_df["topic"].value_counts()
        for tid in range(model.k):
            count = counts.get(tid, 0)
            rows.append({"subject": subject, "year": year, "topic_id": tid,
                         "doc_count": count, "total_docs_year": len(year_df),
                         "proportion": round(count / len(year_df), 6) if len(year_df) > 0 else 0,
                         "top_words": ", ".join(global_tw[tid])})

    pd.DataFrame(rows).to_csv(RESULT_DIR / subject / "topic_prevalence.csv", index=False)
    print(f"  {subject.upper()}: Saved topic_prevalence.csv")

  CS: Saved topic_prevalence.csv
  MATH: Saved topic_prevalence.csv
  PHYSICS: Saved topic_prevalence.csv


## Topic Word Evolution (Native DTM)

In [6]:
all_topic_words_per_year = {}

for subject in LIST_SUBJECT:
    model = all_models[subject]
    years = all_years[subject]
    year_to_tp = all_year_to_tp[subject]

    topic_words_per_year = {}
    rows = []
    for year in years:
        tw = get_topic_words_at_timepoint(model, timepoint=year_to_tp[year], top_n=TOP_N_WORDS)
        for tid, words in tw.items():
            topic_words_per_year[(year, tid)] = words
            rows.append({"subject": subject, "year": year,
                         "topic_id": tid, "top_words": ", ".join(words)})

    all_topic_words_per_year[subject] = topic_words_per_year
    pd.DataFrame(rows).to_csv(RESULT_DIR / subject / "topic_word_evolution.csv", index=False)
    print(f"  {subject.upper()}: Saved topic_word_evolution.csv ({len(rows)} rows)")

  CS: Saved topic_word_evolution.csv (1300 rows)
  MATH: Saved topic_word_evolution.csv (1300 rows)
  PHYSICS: Saved topic_word_evolution.csv (1300 rows)


## Per-Year Coherence & IRBO

In [7]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    topic_words_per_year = all_topic_words_per_year[subject]

    print(f"\n{'='*70}")
    print(f"Per-Year Metrics: {subject.upper()} ({model.k} topics)")
    print(f"{'='*70}")

    # Build corpus-level tokenized texts and dictionary (full corpus as reference)
    corpus_texts_tokenized = df["tokens"].tolist()
    corpus_dictionary = Dictionary(corpus_texts_tokenized)
    print(f"  Corpus: {len(corpus_texts_tokenized):,} docs, {len(corpus_dictionary):,} vocab")

    rows = []
    for year in years:
        year_df = df[df["year"] == year]
        active_topics = sorted(year_df["topic"].unique())
        year_tw = [topic_words_per_year[(year, tid)] for tid in active_topics
                   if (year, tid) in topic_words_per_year]

        valid_tw = [tw for tw in year_tw if len(tw) >= 2]
        if valid_tw:
            cm = CoherenceModel(topics=valid_tw, texts=corpus_texts_tokenized,
                                dictionary=corpus_dictionary, coherence='c_v', processes=7)
            coherence = cm.get_coherence()
        else:
            coherence = 0.0

        irbo = calculate_irbo(year_tw, p=RBO_P)
        quality = 2 * coherence * irbo / (coherence + irbo) if (coherence + irbo) > 0 else 0.0

        print(f"  {year}: {len(year_df):>6,} docs | {len(active_topics):>3} topics | "
              f"Q={quality:.4f} (C={coherence:.4f}, IRBO={irbo:.4f})")

        rows.append({"subject": subject, "year": year, "num_docs": len(year_df),
                     "num_topics_total": model.k, "num_topics_active": len(active_topics),
                     "coherence_cv": round(coherence, 6), "irbo_mean": round(irbo, 6),
                     "topic_quality": round(quality, 6)})

    pd.DataFrame(rows).to_csv(RESULT_DIR / subject / "per_year_metrics.csv", index=False)
    print(f"  Saved: per_year_metrics.csv")


Per-Year Metrics: CS (50 topics)
  Corpus: 165,756 docs, 146,603 vocab
  2000:    488 docs |  49 topics | Q=0.5255 (C=0.4311, IRBO=0.6727)
  2001:    594 docs |  50 topics | Q=0.5078 (C=0.3705, IRBO=0.8067)
  2002:    648 docs |  50 topics | Q=0.5266 (C=0.4062, IRBO=0.7485)
  2003:    825 docs |  50 topics | Q=0.4990 (C=0.3659, IRBO=0.7846)
  2004:    948 docs |  50 topics | Q=0.5178 (C=0.3832, IRBO=0.7981)
  2005:  1,000 docs |  50 topics | Q=0.5228 (C=0.4095, IRBO=0.7230)
  2006:  1,000 docs |  50 topics | Q=0.5410 (C=0.4120, IRBO=0.7875)
  2007:  1,000 docs |  50 topics | Q=0.5465 (C=0.4181, IRBO=0.7885)
  2008:  1,000 docs |  50 topics | Q=0.5278 (C=0.3969, IRBO=0.7874)
  2009:  1,000 docs |  49 topics | Q=0.5314 (C=0.3938, IRBO=0.8166)
  2010:  1,362 docs |  50 topics | Q=0.5190 (C=0.3725, IRBO=0.8556)
  2011:  1,622 docs |  50 topics | Q=0.5203 (C=0.3665, IRBO=0.8966)
  2012:  2,254 docs |  50 topics | Q=0.5077 (C=0.3495, IRBO=0.9273)
  2013:  2,719 docs |  50 topics | Q=0.4956 

## Topic Trends

In [8]:
from scipy.stats import linregress

for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    global_tw = get_aggregated_topic_words(model, top_n=5)

    rows = []
    for tid in range(model.k):
        topic_df = df[df["topic"] == tid]
        if len(topic_df) == 0:
            continue

        year_counts = topic_df["year"].value_counts().sort_index()
        total_per_year = df["year"].value_counts().sort_index()
        proportions = (year_counts / total_per_year).fillna(0)

        # Align proportions with all years
        all_years = sorted(total_per_year.index)
        prop_aligned = proportions.reindex(all_years, fill_value=0.0)

        years_arr = np.array(all_years, dtype=float)
        props_arr = prop_aligned.values.astype(float)

        # Linear regression
        slope, intercept, r_val, p_val, std_err = linregress(years_arr, props_arr)

        # Early/late for display
        topic_years = sorted(year_counts.index)
        early_mean = proportions[topic_years[:5]].mean() if len(topic_years) >= 5 else proportions.mean()
        late_mean = proportions[topic_years[-5:]].mean() if len(topic_years) >= 5 else proportions.mean()

        # Classify by slope significance
        if p_val < 0.05 and slope > 0:
            trend_label = "GROWING"
        elif p_val < 0.05 and slope < 0:
            trend_label = "DECLINING"
        else:
            trend_label = "STABLE"

        top_words = global_tw.get(tid, ["?"])
        rows.append({
            "subject": subject, "topic_id": tid,
            "top_words": ", ".join(top_words),
            "total_docs": len(topic_df),
            "first_year": year_counts.index.min(),
            "last_year": year_counts.index.max(),
            "early_proportion": round(early_mean, 6),
            "late_proportion": round(late_mean, 6),
            "slope": round(slope, 8),
            "r_squared": round(r_val**2, 4),
            "p_value": round(p_val, 6),
            "trend": trend_label,
        })

    trends_df = pd.DataFrame(rows)
    trends_df.to_csv(RESULT_DIR / subject / "topic_trends.csv", index=False)

    g = len(trends_df[trends_df["trend"] == "GROWING"])
    s = len(trends_df[trends_df["trend"] == "STABLE"])
    d = len(trends_df[trends_df["trend"] == "DECLINING"])
    print(f"  {subject.upper()}: Growing={g}, Stable={s}, Declining={d}")


  CS: Growing=14, Stable=12, Declining=24
  MATH: Growing=7, Stable=32, Declining=11
  PHYSICS: Growing=6, Stable=36, Declining=8


## Top 5 Growing & Declining Topics

In [9]:
for subject in LIST_SUBJECT:
    trends_df = pd.read_csv(RESULT_DIR / subject / "topic_trends.csv")

    print(f"\n{'='*80}")
    print(f"  {subject.upper()}")
    print(f"{'='*80}")

    growing = trends_df[trends_df['trend'] == 'GROWING'].sort_values('slope', ascending=False)
    declining = trends_df[trends_df['trend'] == 'DECLINING'].sort_values('slope', ascending=True)

    print(f"\n  " + chr(0x1F4C8) + " TOP 5 GROWING (steepest positive slope):")
    for _, row in growing.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | slope={row['slope']:+.6f} R\u00B2={row['r_squared']:.3f} | "
              f"{row['early_proportion']:.4f} \u2192 {row['late_proportion']:.4f} | {row['top_words']}")

    print(f"\n  " + chr(0x1F4C9) + " TOP 5 DECLINING (steepest negative slope):")
    for _, row in declining.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | slope={row['slope']:+.6f} R\u00B2={row['r_squared']:.3f} | "
              f"{row['early_proportion']:.4f} \u2192 {row['late_proportion']:.4f} | {row['top_words']}")



  CS

  📈 TOP 5 GROWING (steepest positive slope):
    T 23 | slope=+0.002383 R²=0.868 | 0.0345 → 0.0880 | graph, multi, error, local, interaction
    T  3 | slope=+0.002048 R²=0.673 | 0.0325 → 0.0735 | dataset, language, communication, machine, inference
    T 24 | slope=+0.001986 R²=0.925 | 0.0124 → 0.0502 | optimization, object, key, resource, robot
    T  0 | slope=+0.001922 R²=0.635 | 0.0308 → 0.0634 | representation, non, deep, neural_network, policy
    T 16 | slope=+0.001839 R²=0.354 | 0.0137 → 0.0587 | cost, efficiency, datum, self, capability

  📉 TOP 5 DECLINING (steepest negative slope):
    T 13 | slope=-0.002864 R²=0.264 | 0.1091 → 0.0276 | network, datum, extensive, global, well
    T 27 | slope=-0.002315 R²=0.759 | 0.0913 → 0.0451 | algorithm, prediction, environment, test, online
    T  4 | slope=-0.001135 R²=0.787 | 0.0291 → 0.0057 | type, retrieval, channel, logic, theory
    T  1 | slope=-0.001080 R²=0.164 | 0.0451 → 0.0396 | code, space, scale, human, effective
  

## Evolution Summary

In [10]:
for subject in LIST_SUBJECT:
    metrics_df = pd.read_csv(RESULT_DIR / subject / "per_year_metrics.csv")
    trends_df = pd.read_csv(RESULT_DIR / subject / "topic_trends.csv")

    g = len(trends_df[trends_df["trend"] == "GROWING"])
    s = len(trends_df[trends_df["trend"] == "STABLE"])
    d = len(trends_df[trends_df["trend"] == "DECLINING"])

    summary = {"subject": subject, "num_topics": all_models[subject].k,
               "num_years": len(metrics_df),
               "coherence_mean": round(metrics_df["coherence_cv"].mean(), 6),
               "coherence_std": round(metrics_df["coherence_cv"].std(), 6),
               "irbo_mean": round(metrics_df["irbo_mean"].mean(), 6),
               "irbo_std": round(metrics_df["irbo_mean"].std(), 6),
               "quality_mean": round(metrics_df["topic_quality"].mean(), 6),
               "quality_std": round(metrics_df["topic_quality"].std(), 6),
               "topics_growing": g, "topics_stable": s, "topics_declining": d}

    pd.DataFrame([summary]).to_csv(RESULT_DIR / subject / "evolution_summary.csv", index=False)
    print(f"  {subject.upper()}: C={summary['coherence_mean']:.4f}, "
          f"IRBO={summary['irbo_mean']:.4f}, Q={summary['quality_mean']:.4f} | "
          f"↑{g} →{s} ↓{d}")

  CS: C=0.3629, IRBO=0.8836, Q=0.5103 | ↑14 →12 ↓24
  MATH: C=0.3869, IRBO=0.7950, Q=0.5131 | ↑7 →32 ↓11
  PHYSICS: C=0.3723, IRBO=0.8183, Q=0.5060 | ↑6 →36 ↓8
